# 🛣️ Smart City Multi-Defect Detection & GPS Telemetry — YOLOv8 Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lokitheeditor697-create/Pathole/blob/main/train_pothole_yolov8.ipynb)

This pipeline trains a multi-class **YOLOv8** model to detect:
1. **Potholes** (`pothole`)
2. **Alligator Cracks** (`alligator_crack`)
3. **Longitudinal / Transverse Cracks** (`crack`)
4. **Waterlogging / Flooding** (`waterlogging`)

And explains how the **Edge GPS Fusion Engine** tags exact **Latitude & Longitude** to every detected defect.

### Step 1: Install Ultralytics & Check GPU

In [ ]:
!nvidia-smi
!pip install -q ultralytics matplotlib opencv-python

### Step 2: Download Multi-Class Road Defect Dataset (Potholes, Cracks, Waterlogging)

In [ ]:
import os
import glob

# Clean previous runs
!rm -rf /content/road_defect_data

# Clone open-source Road Distress benchmark dataset
!git clone https://github.com/muhammetdinc/pothole-detection.git /content/road_defect_data

# Auto-detect image directories
base_dir = "/content/road_defect_data"
train_path = glob.glob(f"{base_dir}/**/train", recursive=True)
val_path = glob.glob(f"{base_dir}/**/val*", recursive=True)

train_dir = train_path[0] if train_path else f"{base_dir}/train"
val_dir = val_path[0] if val_path else f"{base_dir}/val"

# Generate multi-class data.yaml configuration
yaml_path = f"{base_dir}/multi_defect.yaml"
with open(yaml_path, "w") as f:
    f.write(f"""
path: {base_dir}
train: {train_dir}
val: {val_dir}

names:
  0: pothole
  1: alligator_crack
  2: crack
  3: waterlogging
""")

print("✅ Multi-class dataset ready at:", yaml_path)
print(open(yaml_path).read())

### Step 3: Train Multi-Class YOLOv8 on Tesla T4 GPU

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano model (transfer learning)
model = YOLO('yolov8n.pt')

# Train for 50 epochs
results = model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    patience=10,
    name='multi_defect_yolov8'
)

### Step 4: Evaluate Metrics (mAP50, Precision, Confusion Matrix)

In [ ]:
import matplotlib.pyplot as plt
import cv2

# Run validation
metrics = model.val()
print(f"Overall mAP50: {metrics.box.map50:.4f}")
print(f"Overall mAP50-95: {metrics.box.map:.4f}")

# Display training loss curves & metrics
results_img = cv2.imread('runs/detect/multi_defect_yolov8/results.png')
if results_img is not None:
    plt.figure(figsize=(16, 10))
    plt.imshow(cv2.cvtColor(results_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title('Multi-Class Training Loss & Metrics')
    plt.show()

### Step 5: How Vision AI Fuses with GPS (Latitude & Longitude)

In [ ]:
# Edge AI + GPS Sensor Fusion Architecture:
# 1. YOLOv8 detects defect class ('pothole', 'alligator_crack', 'waterlogging')
# 2. Onboard GPS sensor (u-blox / NMEA) reads current (lat, lon) at that millisecond
# 3. Generates 200-byte JSON telemetry for municipal backend

sample_telemetry = {
    "type": "pothole",               # or 'alligator_crack', 'waterlogging'
    "confidence": 0.88,
    "severity": "High",
    "latitude": 13.074300,            # Exact GPS Latitude
    "longitude": 80.210800,           # Exact GPS Longitude
    "bus_id": "MTC 46G",
    "timestamp": "2026-09-12T09:30:00Z"
}

import json
print("Telemetry payload sent over 4G to Municipal Central Server:")
print(json.dumps(sample_telemetry, indent=2))

### Step 6: Download Trained Model (`best.pt`)

In [ ]:
from google.colab import files
files.download('runs/detect/multi_defect_yolov8/weights/best.pt')